# NFL pick'em odds

Scrapes the Las Vegas odds table from vegasinsider.com and averages the point spread across books.
Negative spread = favored (the more negative, the bigger the expected win).

The site's HTML is flaky — columns get scrambled, junk like `--4.5` shows up, and the spread /
total / moneyline sections are all stacked into one table — so the parsing below validates every
cell and silently drops whatever doesn't make sense instead of crashing.

In [1]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', None)

In [2]:
url = "https://www.vegasinsider.com/nfl/odds/las-vegas/"
tab = pd.read_html(url)[0]

In [3]:
def parse_line(cell):
    """Pull the leading number out of a cell like '+3.5 -110 +'.
    NaN for anything malformed (e.g. the '--4.5' junk the site sometimes renders)."""
    if not isinstance(cell, str):
        return np.nan
    tok = cell.split()[0]
    if tok.upper() in ('PK', 'PICK', 'EVEN'):
        return 0.0
    if re.fullmatch(r'[+-]?\d+(?:\.\d+)?', tok):
        return float(tok)
    return np.nan


def load_sections(tab):
    """The page stacks the spread / total / moneyline sections into one table, with
    rotation numbers restarting at each section. Split them apart and parse every cell.
    Book columns are auto-detected, so no more hand-maintaining the sources list."""
    sources = [c for c in tab.columns
               if c not in ('Time', 'Open') and not str(c).startswith('Unnamed')]
    rows, seen, section = [], set(), 0
    for _, r in tab.iterrows():
        t = r['Time']
        if not isinstance(t, str):
            continue
        m = re.match(r'^(\d+)\s+(.*\S)', t)      # team rows look like '451 Patriots'
        if not m:
            continue                              # skips 'Matchup', 'Final', header junk
        rot = int(m.group(1))
        if rot in seen:                           # rotation number repeated -> new section
            section += 1
            seen = set()
        seen.add(rot)
        rows.append({'section': section, 'Team': m.group(2),
                     **{s: parse_line(r[s]) for s in sources}})
    return pd.DataFrame(rows), sources


def drop_sign_flips(df, sources):
    """The site sometimes renders a book's column with the two teams swapped.
    NaN out any value whose sign disagrees with the row median across books."""
    df = df.reset_index(drop=True).copy()
    med = df[sources].median(axis=1)
    for s in sources:
        flipped = (df[s] * med) < 0
        if flipped.any():
            print(f"{s}: ignoring sign-flipped lines for {df.loc[flipped, 'Team'].tolist()}")
        df.loc[flipped, s] = np.nan
    return df


def clean_spreads(df, sources):
    """Games are consecutive row pairs, and a book's two lines in a game must be
    mirror images (+3.5 / -3.5). Anything else is a bad render -> NaN both sides."""
    df = df.reset_index(drop=True).copy()
    for g in range(0, len(df) - 1, 2):
        for s in sources:
            a, b = df.loc[g, s], df.loc[g + 1, s]
            if np.isnan(a) or np.isnan(b) or a != -b:
                df.loc[[g, g + 1], s] = np.nan
    df = drop_sign_flips(df, sources)
    return df.dropna(subset=sources, how='all')   # drops games already final

In [4]:
full, sources = load_sections(tab)

# classify sections by typical magnitude: spreads are small, moneylines are +-100 and up,
# and totals ('o47.5 ...') never parse at all
med_abs = full.groupby('section')[sources].apply(lambda d: d.abs().median().median())
spread_sections = med_abs[med_abs < 50].index
ml_sections = med_abs[med_abs >= 100].index

spreads = clean_spreads(full[full['section'].isin(spread_sections)], sources)
spreads['ave_spread'] = spreads[sources].mean(axis=1)
spreads['n_books'] = spreads[sources].notna().sum(axis=1)
spreads[['Team'] + sources + ['ave_spread', 'n_books']].sort_values('ave_spread')

BetMGM: ignoring sign-flipped lines for ['Packers', 'Vikings']
HardRock: ignoring sign-flipped lines for ['Buccaneers', 'Bengals', 'Packers', 'Vikings']


,Team,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,ave_spread,n_books
21,Chargers,-10.5,-10.5,-10.5,-10.5,-10.5,-10.5,-10.0,-10.5,-10.5,-10.444444,9
5,Jaguars,-7.5,-7.5,-7.5,-7.5,-7.5,-8.5,-8.5,-7.5,-7.5,-7.722222,9
19,Lions,-6.5,-7.0,-7.0,-7.0,-7.0,-7.5,-7.0,-7.0,-7.0,-7.000000,9
25,Eagles,-4.5,-4.5,-4.5,-4.5,-5.5,NaN,-5.0,-5.0,-4.5,-4.750000,8
23,Raiders,-3.5,-3.5,-3.5,-3.5,-3.5,NaN,-4.0,-3.5,-3.5,-3.562500,8
7,Bengals,-3.5,-3.5,-3.5,-3.5,-3.5,NaN,-4.0,-3.5,-3.5,-3.562500,8
8,Ravens,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.500000,9
1,Seahawks,-3.5,-3.5,-3.5,-3.5,-3.5,NaN,-3.5,-3.5,-3.5,-3.500000,8
3,Rams,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.500000,9
11,Steelers,-3.5,-3.0,-3.0,-3.0,-3.0,NaN,-3.5,-3.0,-3.0,-3.125000,8


## Moneyline → implied win probability

The moneyline section of the same table is literally the odds a team wins the matchup.
Averaged across books, converted to a probability, with the vig removed by normalizing
each game's two probabilities to sum to 1.

In [5]:
ml = drop_sign_flips(full[full['section'].isin(ml_sections)], sources)
ml = ml.dropna(subset=sources, how='all').reset_index(drop=True)
ml['ave_ml'] = ml[sources].mean(axis=1)

def implied_prob(m):
    return 100 / (m + 100) if m > 0 else -m / (-m + 100)

ml['q'] = ml['ave_ml'].map(implied_prob)
for g in range(0, len(ml) - 1, 2):
    tot = ml.loc[g, 'q'] + ml.loc[g + 1, 'q']
    ml.loc[[g, g + 1], 'win_prob'] = ml.loc[[g, g + 1], 'q'] / tot
ml[['Team', 'ave_ml', 'win_prob']].sort_values('win_prob', ascending=False)

Caesars: ignoring sign-flipped lines for ['Packers']
HardRock: ignoring sign-flipped lines for ['Jaguars', 'Falcons', 'Bears']
Fanatics: ignoring sign-flipped lines for ['Packers']


,Team,ave_ml,win_prob
21,Chargers,-589.444444,0.818116
5,Jaguars,-396.875000,0.766316
19,Lions,-337.777778,0.738256
25,Eagles,-225.111111,0.664030
7,Bengals,-199.111111,0.639465
23,Raiders,-196.444444,0.634036
3,Rams,-196.555556,0.631098
1,Seahawks,-190.666667,0.629184
8,Ravens,-189.000000,0.625036
11,Steelers,-168.888889,0.602307
